# React — Reducers

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> The first lesson of this topic has no React in it at all — a reducer is a plain function, and
> that is the point. You will write real, reusable code here, and the same reducer comes back
> in topic 26 when you learn to test it.

## LESSON 52 — A reducer is a pure function

Before any React API, the idea. A **reducer** is a function that takes the current state and a
description of something that happened, and returns the next state:

```js
function reducer(state, action) {
  // ...
  return nextState;
}
```

That is the whole signature, and you have seen its shape twice already:

- `Array.prototype.reduce` from the JavaScript course takes an accumulator and an item and
  returns the next accumulator. Same shape, same name, same idea.
- LESSON 33's `validate(values)` was a pure function of its input. A reducer is pure too, and
  for the same reasons.

### The two rules

**1. It must be pure.** Same `(state, action)` in, same `nextState` out — every time. No
`fetch`, no `Date.now()`, no `Math.random()`, no writing to anything outside it, and above all
**no mutating the state it was given**:

```js
// wrong - mutates the state it was handed
state.tasks.push(newTask);
return state;

// right - returns a new object (LESSON 27-28)
return { ...state, tasks: [...state.tasks, newTask] };
```

That is LESSON 27 and LESSON 28 with a new name on it. If the reducer mutates, everything
downstream that compares states breaks, exactly as it did for `useState`.

**2. An action describes what happened, not what to change.** This is the part people get
wrong, and it is the difference between a reducer that ages well and one that does not:

```js
{ type: "task_added", text: "Buy milk" }        // something happened
{ type: "set_tasks", tasks: [ ...one hundred ] }  // an instruction to the state
```

The first names an event in your application's language. The second is just an assignment
wearing a costume — if every action is `set_x`, you have reimplemented `useState` with extra
ceremony.

### The shape of one

```js
const initialState = { tasks: [], filter: "all" };

function tasksReducer(state, action) {
  switch (action.type) {
    case "task_added":
      return { ...state, tasks: [...state.tasks, { id: action.id, text: action.text, done: false }] };

    case "task_toggled":
      return {
        ...state,
        tasks: state.tasks.map((t) => (t.id === action.id ? { ...t, done: !t.done } : t)),
      };

    case "filter_changed":
      return { ...state, filter: action.filter };

    default:
      throw new Error(`Unknown action: ${action.type}`);
  }
}
```

Three things worth noticing.

**The `switch` is a lookup.** You met this shape in LESSON 24, where one handler served many
buttons by looking up what to do. A reducer is that idea applied to state: one function, many
actions, chosen by a `type`.

**Every branch returns a new object.** No branch mutates, and the parts that did not change are
carried across by the spread — untouched items in `tasks` are literally the same objects
(LESSON 28).

**The `default` throws.** A typo in an action type is otherwise the quietest bug in the
codebase: nothing happens, no error, and you look everywhere except the spelling. Throwing
turns a silent no-op into a loud, immediate message.

### Why bother, when `useState` works

Nothing here replaces `useState`, and LESSON 53 covers when to choose which. What a reducer
buys you is that **all the ways your state can change live in one place, written as plain
JavaScript**. Which means you can read them together, and — as topic 26 will show — you can test
every one of them without rendering anything.

### Key Notes

- A reducer is `(state, action) => nextState`, and it must be **pure**.
- Never mutate the state it is given; return a new object (LESSON 27-28).
- An action **describes what happened**, in your app's language — not an instruction to set a
  value.
- Throw on an unknown action type. A silent typo is far more expensive than a crash.

### Example

**Runnable — plain JS.** No React anywhere. This is the reducer you will use in LESSON 53 and
test in topic 26, so it is worth writing properly now.

In [ ]:
const l52initial = { tasks: [], filter: "all" };

function l52reducer(state, action) {
  switch (action.type) {
    case "task_added":
      return {
        ...state,
        tasks: [...state.tasks, { id: action.id, text: action.text, done: false }],
      };

    case "task_toggled":
      return {
        ...state,
        tasks: state.tasks.map((task) =>
          task.id === action.id ? { ...task, done: !task.done } : task,
        ),
      };

    case "task_removed":
      return { ...state, tasks: state.tasks.filter((task) => task.id !== action.id) };

    case "filter_changed":
      return { ...state, filter: action.filter };

    default:
      throw new Error(`Unknown action: ${action.type}`);
  }
}

// Drive it like a little history of events.
let l52state = l52initial;
const l52history = [
  { type: "task_added", id: "a", text: "Write the brief" },
  { type: "task_added", id: "b", text: "Book the room" },
  { type: "task_toggled", id: "a" },
  { type: "filter_changed", filter: "done" },
];

for (const action of l52history) {
  l52state = l52reducer(l52state, action);
  console.log(action.type.padEnd(16), "->", JSON.stringify(l52state));
}

console.log("");
console.log("initial state untouched?", JSON.stringify(l52initial) === '{"tasks":[],"filter":"all"}');

The last line matters as much as the rest: after four actions the original state object is
exactly as it started. A reducer never edits — it answers.

### Exercise

**Part 1 — in the notebook.** Extend `l52reducer` without breaking either rule.

1. Add `"task_edited"` — change one task's `text`, leaving everything else alone.
2. Add `"completed_cleared"` — remove every task whose `done` is `true`.
3. Add `"all_toggled"` — if every task is done, mark them all undone; otherwise mark them all
   done. (The decision is made *from the state*, which is exactly what a reducer is for.)
4. Prove purity for all three: run each action twice on the same starting state and check the
   two results are equal, and check the starting state is unchanged afterwards.

**Part 2 — judgement, in comments.** Here are six actions. Three describe what happened and
three are instructions in disguise. Sort them, and rewrite the three bad ones.

```js
{ type: "set_tasks", tasks: [] }
{ type: "task_added", id: "x", text: "Hello" }
{ type: "set_filter", filter: "done" }
{ type: "filter_changed", filter: "done" }
{ type: "increment_count_by", amount: 1 }
{ type: "task_removed", id: "x" }
```

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Four reducers, each broken in a different way. Say what is wrong with each and what it would
cost you in a real app.

```js
// A
function reducer(state, action) {
  if (action.type === "task_added") {
    state.tasks.push({ id: action.id, text: action.text });
  }
  return state;
}

// B
function reducer(state, action) {
  switch (action.type) {
    case "task_added":
      return { ...state, tasks: [...state.tasks, { id: crypto.randomUUID(), text: action.text }] };
    default:
      return state;
  }
}

// C
function reducer(state, action) {
  switch (action.type) {
    case "task_added":
      return { ...state, tasks: [...state.tasks, action.task] };
    // no default
  }
}

// D
function reducer(state, action) {
  switch (action.type) {
    case "tasks_loaded":
      return { ...state, tasks: action.tasks };
    default:
      return state;
  }
}
```

Then answer: **D** looks like the `set_tasks` anti-pattern from the exercise, and it is
actually fine. What is the difference, and what does that tell you about the rule?

In [ ]:
// Your code here

## LESSON 53 — `useReducer` in a component

LESSON 52's reducer is a plain function that knows nothing about React. This lesson connects
it, and the connection is one line.

### The API

> `useReducer` is a React Hook that lets you add a reducer to your component.

```jsx
const [state, dispatch] = useReducer(tasksReducer, initialState);
```

It returns a pair, exactly like `useState`:

> 1. The current state.
> 2. The `dispatch` function that lets you update the state to a different value and trigger a
>    re-render.

And you send an action by calling `dispatch`:

```jsx
dispatch({ type: "task_added", id: crypto.randomUUID(), text });
```

> You need to pass the action as the only argument to the `dispatch` function.
> `dispatch` functions do not have a return value.

In the playground experiment the reducer is **copied unchanged** from the notebook cell. It
knew nothing about React then and it knows nothing now — the component only dispatches and
renders.

### `dispatch` behaves exactly like a setter

Everything you learned in LESSON 26 applies without modification:

> The `dispatch` function **only updates the state variable for the *next* render**. If you read
> the state variable after calling the `dispatch` function, you will still get the old value
> that was on the screen before your call.

```js
function handleClick() {
  console.log(state.age);                 // 42
  dispatch({ type: "incremented_age" });
  console.log(state.age);                 // Still 42!
}
```

Same fixed-per-render value, same batching, same reason. `dispatch` is a setter that takes an
event instead of a value.

### Where the id comes from

Notice the component generates the id and puts it **in the action**:

```jsx
dispatch({ type: "task_added", id: crypto.randomUUID(), text });
```

not inside the reducer. That is LESSON 52's purity rule doing real work: a reducer that called
`crypto.randomUUID()` itself would produce a different result for the same input, and React
would call it twice and get two different ids.

Which it does. Measured in the playground, one click of **add**:

```text
   dispatch — task_added "Write the brief"
   reducer — task_added
   reducer — task_added
```

> In Strict Mode, React will **call your reducer and initializer twice** in order to help you
> find accidental impurities. This is development-only behavior and does not affect production.
> If your reducer and initializer are pure (as they should be), this should not affect your
> logic. The result from one of the calls is ignored.

A pure reducer gives the same answer both times and nothing is affected. An impure one produces
two different answers, and you have just discovered a bug you would otherwise have shipped.

### When to prefer it over `useState`

Not "when state gets complicated" — that is vague enough to be useless. Three concrete signals:

**1. Several pieces of state change together.** If one event always updates three values, a
reducer keeps that transition in one place instead of three setter calls that must not drift
apart.

**2. The next state depends on the current state in a non-trivial way.** `all_toggled` from
LESSON 52 had to look at every task before deciding. That logic has to live somewhere; in a
reducer it lives with the rest of the transitions.

**3. You want to read all the ways the state can change, together.** React's own framing:

> `useReducer` … lets you move the state update logic from event handlers into a single
> function outside of your component.

And one more that the course has already earned: **you can test it without React**, which is
what you did in LESSON 52 and will do again in topic 26.

Stay with `useState` when a value is independent and changes on its own — a controlled input's
text, whether a panel is open, a loading flag. The experiment uses **both**: `useReducer` for
the tasks and filter, `useState` for the text being typed. That mix is normal and correct.

### Key Notes

- `const [state, dispatch] = useReducer(reducer, initialState)` — a pair, like `useState`.
- `dispatch(action)` returns nothing and updates state **for the next render**, exactly like a
  setter (LESSON 26).
- Generate ids and timestamps **in the action**, never in the reducer.
- Strict Mode calls the reducer twice to check it is pure. A pure one does not notice.
- Reach for it when several values change together, or when transitions are worth reading in
  one place. Keep `useState` for independent values.

### Example

**In the playground.** No cell — LESSON 52 already ran the reducer by hand, and what is new
here is the wiring, which needs a real component.

Point `playground/src/App.jsx` at `./experiments/24-reducer.jsx` and open the console. Add a
task and read the three lines: one dispatch, two reducer calls. Then toggle and filter.

### Exercise

**In the playground**, in `24-reducer.jsx`.

1. Add a task with the console open. Write down the order of `dispatch`, `reducer` and `render`
   lines. Why are there two reducer calls and only one dispatch?
2. Log `state.tasks.length` immediately **after** the `dispatch` call inside `handleSubmit`.
   What does it print for the very first task, and which lesson explains it?
3. Move the id generation into the reducer — `id: crypto.randomUUID()` inside the
   `task_added` case, and remove it from the action. Add two tasks and inspect the list in the
   React DevTools Components tab. What has gone wrong, and why does Strict Mode make it obvious?
4. Put it back. Now add a `"cleared"` action that empties the list, wire it to a button, and
   confirm the filter buttons still work afterwards.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

For each situation, say `useState`, `useReducer`, or **both**, and why in one line.

1. A single checkbox that toggles dark mode.
2. A multi-step wizard with a current step, collected answers, and a validation error per step.
3. The text in a search box.
4. A shopping basket: items, quantities, a discount code, and a computed total.
5. Whether a dropdown is open.
6. A drawing canvas with an undo history.

Then answer:

- Number 4 mentions a computed total. Which lesson says that should not be in the reducer's
  state at all, and where does it belong?
- Number 6 is the one where a reducer is not just tidier but genuinely enables something
  `useState` makes hard. What is it, and which property of a reducer makes it possible?

In [ ]:
// Your code here

## LESSON 54 — Lazy init, grouped updates, and naming

Three smaller things that make reducers pleasant to live with.

### Lazy initialisation

`useReducer` takes an optional third argument:

```js
const [state, dispatch] = useReducer(reducer, initialArg, init?);
```

> **optional** `init`: The initializer function that should return the initial state. If it's
> not specified, the initial state is set to `initialArg`. Otherwise, the initial state is set
> to the result of calling `init(initialArg)`.

Why it exists:

> Although the result of `createInitialState(username)` is only used for the initial render,
> you're still calling this function on every render. This can be wasteful if it's creating
> large arrays or performing expensive calculations.

```jsx
// called on EVERY render, result thrown away after the first
const [state, dispatch] = useReducer(reducer, createInitialState(username));

// called ONCE
const [state, dispatch] = useReducer(reducer, username, createInitialState);
```

> Notice that you're passing `createInitialState`, which is the *function itself*, and not
> `createInitialState()`, which is the result of calling it.

That distinction is LESSON 22's `onClick={fn}` versus `onClick={fn()}` in a new place — pass
the function, do not call it.

Use it when building the initial state costs something real: parsing, generating a large
structure, reading storage. For `{ tasks: [], filter: "all" }` it would be ceremony, and the
plain second argument is right.

> The initialiser is held to the same standard as the reducer: Strict Mode calls **both** twice
> to check they are pure.

### Grouped updates

This is the return on the whole topic. When one event changes several things, a reducer keeps
the change in one place:

```jsx
// with useState — four setters that must never drift apart
function handleSubmit() {
  setTasks([...tasks, newTask]);
  setText("");
  setError(null);
  setDirty(true);
}

// with a reducer — one event, one transition
dispatch({ type: "task_submitted", id, text });
```

The difference is not typing. It is that in the first version, "what happens when a task is
submitted" is spread across a handler and can be edited in one place and forgotten in another.
In the second it is a single `case` you can read end to end — and test.

It also removes a whole class of bug you met in LESSON 26: with several setters, a reader has to
work out whether they batch, whether any depends on another's new value, and what happens if
one is added later. A reducer answers all of that by construction — the next state is computed
once, from the previous one.

### Naming

Actions are the vocabulary your application is written in. Two conventions carry their weight:

**Name the event, past tense.** `task_added`, `filter_changed`, `tasks_loaded` — not `addTask`,
`setFilter`, `loadTasks`. The past tense is a small trick with a real effect: it is hard to
write `set_tasks` in the past tense, so the naming rule catches the anti-pattern from LESSON 52
before you commit it.

**Keep the payload flat and minimal.** Send what happened, not the whole world:

```js
{ type: "task_toggled", id: "a" }                    // the reducer can find the rest
{ type: "task_toggled", task: { ...everything } }    // now two copies can disagree
```

And name the reducer for what it owns — `tasksReducer`, `formReducer` — because in topic 26 you
will import it by that name in a test file, and in topic 27 the same instinct becomes a *slice*.

### Key Notes

- Pass an `init` **function** as the third argument when building the initial state is
  expensive. Pass the function, not its result.
- A reducer's real payoff is **grouped updates**: one event, one transition, read in one place.
- Name actions as past-tense events, and keep payloads minimal.
- Strict Mode double-calls the initialiser as well as the reducer, for the same reason.

### Example

**Runnable — plain JS.** The reducer from LESSON 52, now with an initialiser and one grouped
action — still with no React anywhere, which is the whole argument for writing them this way.

In [ ]:
// An initialiser: expensive enough to be worth not repeating.
function l54createInitialState(savedText) {
  const tasks = savedText
    .split(",")
    .map((part) => part.trim())
    .filter((part) => part !== "")
    .map((text, index) => ({ id: `s${index}`, text, done: false }));

  return { tasks, filter: "all", draft: "", error: null };
}

function l54reducer(state, action) {
  switch (action.type) {
    // one event, four things changing together
    case "task_submitted": {
      if (action.text.trim() === "") {
        return { ...state, error: "A task needs some text." };
      }
      return {
        ...state,
        tasks: [...state.tasks, { id: action.id, text: action.text.trim(), done: false }],
        draft: "",
        error: null,
      };
    }

    case "draft_changed":
      return { ...state, draft: action.text };

    case "filter_changed":
      return { ...state, filter: action.filter };

    default:
      throw new Error(`Unknown action: ${action.type}`);
  }
}

const l54state0 = l54createInitialState("Write the brief, Book the room");
console.log("initialised:", JSON.stringify(l54state0.tasks.map((t) => t.text)));

const l54state1 = l54reducer(l54state0, { type: "draft_changed", text: "Send invites" });
const l54state2 = l54reducer(l54state1, { type: "task_submitted", id: "n1", text: "Send invites" });

console.log("after submit — tasks:", l54state2.tasks.length,
            "| draft:", JSON.stringify(l54state2.draft),
            "| error:", l54state2.error);

// the empty-text path changes only the error, and adds nothing
const l54state3 = l54reducer(l54state2, { type: "task_submitted", id: "n2", text: "   " });
console.log("empty submit — tasks:", l54state3.tasks.length, "| error:", JSON.stringify(l54state3.error));

One dispatch, three fields updated, and the failure path handled in the same `case` as the
success path — which is exactly the thing that gets forgotten when four setters are spread
through a handler.

### Exercise

**Part 1 — in the notebook.**

1. Add `"error_dismissed"` which clears the error and nothing else.
2. Add `"draft_submitted_with_tag"`, which adds a task **and** sets the filter so the new task
   is visible — one action, two consequences. Decide for yourself what "visible" means and say
   why in a comment.
3. Write `l54replay(initialState, actions)` which folds a list of actions over a starting state
   and returns the final one. Use it to replay a five-action history and log the result.
   (You have written this before — it is `Array.prototype.reduce` with your reducer as the
   callback, which is where the name comes from.)
4. Prove the initialiser is worth it: add a `console.log` inside `l54createInitialState`, call
   it twice directly, and say in a comment what React's third argument would have avoided.

**Part 2 — in the playground.** In `24-reducer.jsx`, replace the separate `useState` for the
text with a `draft` field in the reducer, and make the submit a single `task_submitted` action
like the one above. Then answer in a comment: is this version better? Say honestly what it
gained and what it cost.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Review this reducer as if in a pull request. Find **five** problems.

```js
function reducer(state, action) {
  switch (action.type) {
    case "setUser":
      return { ...state, user: action.user, lastLogin: Date.now() };

    case "addItem":
      state.items.push(action.item);
      return { ...state };

    case "updateTotal":
      return { ...state, total: state.items.reduce((s, i) => s + i.price, 0) };

    case "setLoadingTrue":
      return { ...state, loading: true };

    case "setLoadingFalse":
      return { ...state, loading: false };
  }
}
```

Then answer:

- Two of the five are the *same* problem wearing different clothes. Which two?
- One of the cases should not exist at all, whatever it is renamed to. Which, and which lesson
  says so?

In [ ]:
// Your code here